<a href="https://colab.research.google.com/github/TrungTan369/Machine-Learning-Assignment/blob/main/notebooks/ex3_imageData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
print("Hello world from colab web font-end")

Hello world from colab web font-end


In [ ]:
# Cell 1: Tải dataset từ Kaggle (kagglehub)
# Cài đặt và tải dataset INRIA (chạy trên Colab
%pip install kagglehub -q
import kagglehub, os
print("Downloading INRIA Person from Kaggle...")
dataset_path = kagglehub.dataset_download("jcoral02/inriaperson")
print("Done. dataset_path =", dataset_path)

100%|██████████| 582M/582M [00:09<00:00, 64.2MB/s] 

Extracting files...


Done. dataset_path = /root/.cache/kagglehub/datasets/jcoral02/inriaperson/versions/1


In [7]:
# ...existing code...
# Cell 2: Machine Learning Traditional - EDA, feature-extract (pretrained), save
import zipfile, shutil, os
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Markdown, display

# nếu zip -> giải nén
if isinstance(dataset_path, str) and dataset_path.endswith('.zip'):
    extract_dir = '/content/inriaperson'
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(dataset_path,'r') as z: z.extractall(extract_dir)
    dataset_root = extract_dir
else:
    dataset_root = dataset_path

def find_image_root(root):
    p = Path(root)
    for child in p.iterdir():
        if child.is_dir() and any(child.glob('**/*.jpg')): return str(p)
    for sub in p.rglob('*'):
        if sub.is_dir() and any(sub.glob('*.jpg')): return str(sub)
    return str(p)

dataset_for_loader = find_image_root(dataset_root)
print("Dataset root:", dataset_for_loader)

# Cấu hình
image_size=(224,224); batch_size=32; model_name='resnet50'; pooling='avg'

# ensure local module import
import sys
repo_root = Path().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.image_utils import load_and_preprocess_data, extract_features_pretrained, save_features_to_disk

# --- Load (resize) để dùng cho feature-extract và hiển thị mẫu ---
X_raw, y_raw, class_names = load_and_preprocess_data(dataset_for_loader, image_size=image_size, batch_size=batch_size, shuffle=True)

# --- EDA cơ bản từ dữ liệu đã load ---
n_samples = X_raw.shape[0]
n_classes = len(class_names)
unique, counts = np.unique(y_raw, return_counts=True)

# bảng phân phối nhãn
label_df = pd.DataFrame({
    'class_index': unique,
    'class_name': [class_names[i] for i in unique],
    'count': counts
}).sort_values('class_index').reset_index(drop=True)

# pixel statistics (resized images)
pix_min, pix_max = X_raw.min(), X_raw.max()
pix_mean = X_raw.mean()
pix_std = X_raw.std()
# per-channel mean/std
channel_mean = X_raw.mean(axis=(0,1,2))
channel_std = X_raw.std(axis=(0,1,2))

pixel_stats = pd.DataFrame({
    'metric': ['dtype','min','max','mean','std'],
    'value': [str(X_raw.dtype), float(pix_min), float(pix_max), float(pix_mean), float(pix_std)]
})
chan_stats = pd.DataFrame({
    'channel': ['R','G','B'],
    'mean': channel_mean.round(4),
    'std': channel_std.round(4)
})

# --- EDA bổ sung: thông tin ảnh gốc (không bị resize) ---
orig_sizes = []
for idx, cls in enumerate(class_names):
    cls_dir = Path(dataset_for_loader) / cls
    if not cls_dir.exists(): continue
    for p in cls_dir.rglob('*'):
        if p.suffix.lower() not in ('.jpg','.jpeg','.png'): continue
        try:
            with Image.open(p) as im:
                w,h = im.size
                mode = im.mode
            orig_sizes.append({'path':str(p), 'class':cls, 'width':w, 'height':h, 'mode':mode})
        except Exception:
            continue
orig_df = pd.DataFrame(orig_sizes)
if not orig_df.empty:
    orig_df['aspect'] = (orig_df['width'] / orig_df['height']).round(2)
    size_summary = orig_df.groupby('class').agg(
        n_images = ('path','count'),
        mean_width = ('width','mean'),
        mean_height = ('height','mean'),
        median_aspect = ('aspect','median')
    ).round(1).reset_index()
else:
    size_summary = pd.DataFrame(columns=['class','n_images','mean_width','mean_height','median_aspect'])

# --- Hiển thị bảng và biểu đồ ngắn ---
print("Images:", X_raw.shape, "Labels:", y_raw.shape, "Classes:", class_names)
display(Markdown("### Phân phối nhãn (số ảnh mỗi lớp)"))
display(label_df)
display(Markdown("### Thống kê pixel (sau resize)"))
display(pixel_stats)
display(Markdown("### Thống kê theo kênh màu (sau resize)"))
display(chan_stats)
if not size_summary.empty:
    display(Markdown("### Thống kê kích thước ảnh gốc theo lớp"))
    display(size_summary)

# show samples (1 ảnh mỗi lớp, tối đa 6 lớp)
to_show = min(n_classes,6)
fig,axs = plt.subplots(1,to_show,figsize=(3*to_show,3))
for i in range(to_show):
    idx = (y_raw==unique[i]).nonzero()[0][0]
    axs[i].imshow(X_raw[idx].astype('uint8')); axs[i].axis('off'); axs[i].set_title(f"{class_names[unique[i]]}\ncount={counts[i]}")
plt.tight_layout(); plt.show()

# --- Tự động tạo báo cáo ngắn (markdown) và lưu vào reports/ ---
report_lines = []
report_lines.append(f"# EDA Report - ex3_imageData\n")
report_lines.append(f"- Dataset root: `{dataset_for_loader}`\n")
report_lines.append(f"- Total images (after resize load): **{n_samples}**\n")
report_lines.append(f"- Number of classes: **{n_classes}**\n")
report_lines.append("\n## Label distribution\n")
for _,row in label_df.iterrows():
    report_lines.append(f"- {row['class_name']} ({int(row['class_index'])}): {int(row['count'])} images\n")
report_lines.append("\n## Pixel summary (resized)\n")
report_lines.append(f"- dtype: {X_raw.dtype}\n- min: {pix_min}\n- max: {pix_max}\n- mean: {pix_mean:.4f}\n- std: {pix_std:.4f}\n")
report_lines.append("\n## Channel mean/std (R,G,B)\n")
report_lines.append(f"- mean: {list(channel_mean.round(4))}\n- std: {list(channel_std.round(4))}\n")
if not size_summary.empty:
    report_lines.append("\n## Original image size summary (per class)\n")
    report_lines.append(size_summary.to_markdown(index=False))

reports_dir = Path('reports'); reports_dir.mkdir(exist_ok=True)
report_path = reports_dir / 'ex3_imageData_EDA.md'
report_path.write_text("\n".join(report_lines), encoding='utf-8')

display(Markdown(f"**EDA report saved to** `{report_path}`"))
display(Markdown("----\n**Short analysis / suggestions:**\n- Dataset appears with class imbalance: consider augmentation or class-weighted training if imbalance >2x.\n- Image sizes vary; using resize to consistent size is fine for pretrained CNN.\n- Channel statistics show typical 0-255 range; consider model-specific preprocess (imagenet) before feature extraction.\n"))

# Extract features (pretrained)
X_features = extract_features_pretrained(X_raw, model_name=model_name, batch_size=batch_size, pooling=pooling)
print("Features shape:", X_features.shape)

# Save
prefix = f"inria_{model_name}_{image_size[0]}"
x_path, y_path = save_features_to_disk(X_features, y_raw, prefix)
print("Saved:", x_path, y_path)
# ...existing code...

Dataset root: /root/.cache/kagglehub/datasets/jcoral02/inriaperson/versions/1


ModuleNotFoundError: No module named 'modules'

In [ ]:
# Cell 3: Deep Learning - Transfer learning end-to-end (train/val/test)
import tensorflow as tf
from pathlib import Path

# Tạo tf.data từ thư mục (dùng dataset_for_loader)
img_size = image_size
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_for_loader, labels='inferred', label_mode='int',
    image_size=img_size, batch_size=32, validation_split=0.2, subset='training', seed=42)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_for_loader, labels='inferred', label_mode='int',
    image_size=img_size, batch_size=32, validation_split=0.2, subset='validation', seed=42)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# Build model (transfer learning)
base = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(*img_size,3))
base.trainable = False
inputs = tf.keras.Input(shape=(*img_size,3))
x = tf.keras.applications.resnet.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(train_ds.class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(model.summary())

# Train head
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

# (Optional) Fine-tune some base layers
base.trainable = True
for layer in base.layers[:-20]: layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=5)

# Evaluate and save model
model.evaluate(val_ds)
model.save('models/inria_transfer_resnet50')
print("Saved transfer model to models/inria_transfer_resnet50")